<a href="https://colab.research.google.com/github/naceurarij2-design/Multilingual-Summarization-chatbot/blob/main/03_LoRA_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade torchao -q

In [ ]:
import os
import torch
from google.colab import drive
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import load_from_disk
from peft import LoraConfig, get_peft_model, TaskType

# =====================================================================
# 1. ENVIRONMENT & DIRECTORY CONFIGURATION
# =====================================================================
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/New_Multilingual_AI_Project"
TENSOR_INPUT_PATH = f"{PROJECT_DIR}/processed_outputs/tokenized_dataset_tensor"
OUTPUT_CHECKPOINT_DIR = f"{PROJECT_DIR}/trained_model_checkpoints"
FINAL_ADAPTER_DIR = f"{OUTPUT_CHECKPOINT_DIR}/final_multilingual_adapter"

BASE_MODEL = "facebook/mbart-large-50"

# =====================================================================
# 2. LOAD DATASET & PREPARE TOKENS
# =====================================================================
print("📦 Loading context-aware tokenized tensors from Drive...")
tokenized_dataset = load_from_disk(TENSOR_INPUT_PATH)

# Shuffle the data to ensure balanced language and chronological distribution
tokenized_dataset = tokenized_dataset.shuffle(seed=42)

# Train/Validation split optimized for steady evaluation
dataset_split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
val_dataset = dataset_split["test"]

print(f"📊 Training Samples: {len(train_dataset)} | Validation Samples: {len(val_dataset)}")

# =====================================================================
# 3. LOAD MODEL & OPTIMIZED PEFT-LORA CORE CONFIGURATION
# =====================================================================
print(f"🧠 Loading base model weights: {BASE_MODEL}...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("⚡ Injecting Broad-Spectrum LoRA Target Modules...")
# 🔥 OPTIMIZATION 1: Expanded target modules beyond q and v.
# Targeting q, v, k, and out_proj inside mBART's cross-attention layers
# allows the adapter to learn how to transition naturally between historical text maps.
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=32,                  # Increased rank from 16 to 32 to capture nuanced grammatical flow
    lora_alpha=64,         # Scaled alpha proportionally (2x Rank)
    lora_dropout=0.05,     # Reduced slightly to prevent underfitting on structured prompt templates
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# =====================================================================
# 4. OPTIMIZED TRAINING ARGUMENTS (SFT FLOW Tuning)
# =====================================================================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

# 🔥 OPTIMIZATION 2: Advanced convergence hyperparameters
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_CHECKPOINT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",

    # Tuning learning rate and schedule for deep structural memory retention
    learning_rate=3e-4,                  # Lowered from 5e-4 to prevent model from overriding foundational layers
    lr_scheduler_type="cosine",          # Cosine decay prevents abrupt optimization plateaus
    warmup_ratio=0.1,                    # Dedicates first 10% of training to warm up attention grids safely
    weight_decay=0.05,                   # Higher weight decay forces cleaner, non-repetitive summaries

    num_train_epochs=7,                  # Increased from 5 to solidify connection mechanics
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",        # Tracks global validation loss for ideal checkpoint extraction
    greater_is_better=False,
    report_to="none"
)

# 🔥 OPTIMIZATION 3: Synchronizing Multi-lingual Control Tokens
# Explicitly configures model generation defaults inside the trainer environment
# to keep paragraphs cohesive across foreign language domains.
model.config.max_length = 250
model.config.min_length = 50
model.config.no_repeat_ngram_size = 3
model.config.early_stopping = True

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,   # <-- Updated to matching library standards
    data_collator=data_collator
)
# =====================================================================
# 5. EXECUTE COHERENT TRAINING LOOP & ADAPTER EXPORT
# =====================================================================
print("\n🔥 Starting Production-Aligned LoRA Fine-Tuning...")
trainer.train()

print(f"\n💾 Archiving fine-tuned context-aware LoRA adapters at: {FINAL_ADAPTER_DIR}")
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print("🎉 Phase 3 Complete! Your model is now trained to generate summaries with natural transitions.")